# Bronze 2: split the long PDFs into parts

Task `bronze_split` of the job `ingest_financebench`. Cuts every PDF in `bronze_pdfs` with more than `PART_PAGES`
pages into **parts** of at most `PART_PAGES` pages, and stores the bytes of each part in `bronze_pdf_parts`. A
shorter PDF is one part, with its original bytes. Silver reads the parts, never the whole PDFs.

| Column | Type | Holds |
| --- | --- | --- |
| `path` | STRING | The original PDF |
| `part_no` | INT | 1, 2, 3, ... |
| `page_offset` | INT | Pages before this part. The real page is `page_offset + page_id + 1` |
| `pages` | INT | Pages in this part |
| `content` | BINARY | The bytes of the part (a PDF) |
| `split_at` | TIMESTAMP | When it was cut |

**Why.** The parser throttles (`Request rate limit exceeded`) on a long PDF in one request, and a failed page throws
away the whole parse. A 155-page filing lost 27 pages when parsed whole, and came back complete in four parts. The
parser also rejects PDFs over 500 pages. Cutting does not change the content, so it belongs in bronze: the parts are
still raw bytes.

**Incremental.** Only PDFs without parts are cut, so a rerun does nothing. A PDF that cannot be read fails the
task. The cutting logic is in [splitting.py](./splitting.py), tested with `pytest`.

Written to [doc/code_standard.md](../../doc/code_standard.md).

In [ ]:
%pip install pypdf -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
from pathlib import Path

from pyspark.sql import functions as F, types as T

import splitting  # next to this notebook

In [ ]:
CATALOG = "workspace"
BRONZE_SCHEMA = f"{CATALOG}.finance_bench_bronze"

PDFS_TABLE = f"{BRONZE_SCHEMA}.bronze_pdfs"
PARTS_TABLE = f"{BRONZE_SCHEMA}.bronze_pdf_parts"

PART_PAGES = 50  # PDFs with more pages are cut into parts of this size

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {PARTS_TABLE} (
  path STRING, part_no INT, page_offset INT, pages INT, content BINARY, split_at TIMESTAMP
)""")

## Step 1: the cutting UDF

`splitting.py` is next to the notebook, so the executors cannot import it: `addArtifacts` sends it to them. A plain
UDF, not a pandas UDF: the step below gives each task a single PDF, so there is no batch to vectorise, and a
pandas UDF cannot always return an array of structs.

**Example.** One PDF in, an array of parts out (page counts are illustrative):

| input | `cut(content, 50)` |
| --- | --- |
| `PGE_2020_10K.pdf`, 155 pages | `[{1, 0, 50, 25 50 44 46 …}, {2, 50, 50, …}, {3, 100, 50, …}, {4, 150, 5, …}]` (`part_no`, `page_offset`, `pages`, `content`) |
| `AMD_2022_10K.pdf`, 42 pages | `[{1, 0, 42, <the original bytes>}]` |

In [ ]:
spark.addArtifacts(str(Path("splitting.py").resolve()), pyfile=True)


@F.udf(returnType=T.ArrayType(T.StructType([
    T.StructField("part_no", T.IntegerType()), T.StructField("page_offset", T.IntegerType()),
    T.StructField("pages", T.IntegerType()), T.StructField("content", T.BinaryType()),
])))
def cut(content, size):
    return splitting.split_pdf(bytes(content), size)

## Step 2: cut the PDFs that have no parts

One PDF per task (`repartition`), so the PDFs are cut in parallel on the executors and their bytes go straight into
the table, never through the driver. `optimizeWrite` is off so there is one file per PDF: the parse stream reads a
few files per batch.

**Example.** `bronze_pdfs` holds `3M_2018_10K.pdf` (cut by an earlier run), `PGE_2020_10K.pdf` (155 pages) and
`AMD_2022_10K.pdf` (42 pages).

**2a. After the `left_anti` join.** Only the PDFs without parts:

| path | content |
| --- | --- |
| …/pdfs/PGE_2020_10K.pdf | `25 50 44 46 …` |
| …/pdfs/AMD_2022_10K.pdf | `25 50 44 46 …` |

**2b. After `explode` of `cut`.** One row per part, in the struct `p`:

| path | p |
| --- | --- |
| …/pdfs/PGE_2020_10K.pdf | `{1, 0, 50, …}` |
| …/pdfs/PGE_2020_10K.pdf | `{2, 50, 50, …}` |
| …/pdfs/PGE_2020_10K.pdf | `{3, 100, 50, …}` |
| …/pdfs/PGE_2020_10K.pdf | `{4, 150, 5, …}` |
| …/pdfs/AMD_2022_10K.pdf | `{1, 0, 42, …}` |

**2c. After the last `select`, appended to `bronze_pdf_parts`:**

| path | part_no | page_offset | pages | content | split_at |
| --- | --- | --- | --- | --- | --- |
| …/pdfs/PGE_2020_10K.pdf | 1 | 0 | 50 | `25 50 44 46 …` | 2026-11-20 09:20:11 |
| …/pdfs/PGE_2020_10K.pdf | 2 | 50 | 50 | `25 50 44 46 …` | 2026-11-20 09:20:11 |
| …/pdfs/PGE_2020_10K.pdf | 3 | 100 | 50 | `25 50 44 46 …` | 2026-11-20 09:20:11 |
| …/pdfs/PGE_2020_10K.pdf | 4 | 150 | 5 | `25 50 44 46 …` | 2026-11-20 09:20:11 |
| …/pdfs/AMD_2022_10K.pdf | 1 | 0 | 42 | `25 50 44 46 …` | 2026-11-20 09:20:11 |

When no PDF is new, the count is 0 and nothing is written.

In [ ]:
todo = (
    spark.table(PDFS_TABLE).select("path", "content")
    .join(spark.table(PARTS_TABLE).select("path").distinct(), "path", "left_anti")
)
pdfs = todo.count()
print(f"{pdfs} PDFs to split.")

if pdfs:  # repartition(0) is not allowed
    (
        todo.repartition(pdfs)
        .select("path", F.explode(cut("content", F.lit(PART_PAGES))).alias("p"))
        .select("path", "p.part_no", "p.page_offset", "p.pages", "p.content", F.current_timestamp().alias("split_at"))
        .write.mode("append")
        .option("optimizeWrite", "false")
        .saveAsTable(PARTS_TABLE)
    )

In [ ]:
display(spark.sql(f"SELECT count(DISTINCT path) AS pdfs, count(*) AS parts, max(pages) AS longest_part FROM {PARTS_TABLE}"))